# LeiMai Oracle / Kaggle Matrix Runner

CPU batch mode for feature-native matrix training (1m only).

In [ ]:
REPO_URL = "https://github.com/Le1Ma1/leimai-oracle.git"
GITHUB_TOKEN = ""  # optional: for private clone
GITHUB_USERNAME = "Le1Ma1"  # token owner username (fallback auth principal)
BRANCH = "main"
BATCH_INDEX = 1
BATCH_TOTAL = 3
MAX_ROUNDS = 1
SKIP_INGEST = True
RUN_VALIDATE = True
SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT",
    "DOGEUSDT", "LTCUSDT", "LINKUSDT", "BCHUSDT", "TRXUSDT",
    "ETCUSDT", "XLMUSDT", "EOSUSDT", "XMRUSDT", "ATOMUSDT",
]


In [ ]:
from pathlib import Path
import base64
import os
import shutil
import subprocess

repo_root = Path('/kaggle/working/leimai-oracle')

def _run_clone(cmd: list[str]) -> tuple[int, str, str]:
    proc = subprocess.run(cmd, text=True, capture_output=True)
    return proc.returncode, (proc.stdout or ''), (proc.stderr or '')


def clone_with_auth(repo_url: str, branch: str, target: Path, token: str, username: str) -> None:
    token = (token or '').strip()
    if not token:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(target)], check=True)
        return

    principals = ['x-access-token']
    if username.strip() and username.strip() not in principals:
        principals.append(username.strip())

    errors: list[str] = []
    for principal in principals:
        auth = base64.b64encode(f"{principal}:{token}".encode('utf-8')).decode('ascii')
        cmd = [
            'git',
            '-c', f'http.extraHeader=AUTHORIZATION: basic {auth}',
            'clone', '--depth', '1', '--branch', branch, repo_url, str(target),
        ]
        code, _out, err = _run_clone(cmd)
        if code == 0:
            return
        snippet = '\n'.join((err or '').strip().splitlines()[-4:])
        errors.append(f"{principal}: {snippet or 'unknown clone error'}")

    raise RuntimeError(
        'git clone failed after auth fallback. Verify token scope (Contents: Read), token owner, and repo access. '
        + ' | '.join(errors)
    )

# Clean broken partial clone directory if it exists without .git
if repo_root.exists() and not (repo_root / '.git').exists():
    shutil.rmtree(repo_root, ignore_errors=True)

if not repo_root.exists():
    clone_with_auth(REPO_URL, BRANCH, repo_root, GITHUB_TOKEN, GITHUB_USERNAME)

os.chdir(repo_root)
print('repo_root=', repo_root)


In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'engine/requirements.txt'], check=True)


In [ ]:
def split_batches(symbols, total):
    total = max(1, int(total))
    n = len(symbols)
    out = []
    for i in range(total):
        s = round(i * n / total)
        e = round((i + 1) * n / total)
        bucket = symbols[int(s):int(e)]
        if bucket:
            out.append(bucket)
    return out or [symbols]

batches = split_batches(SYMBOLS, BATCH_TOTAL)
batch_idx = max(1, min(BATCH_INDEX, len(batches)))
batch_symbols = batches[batch_idx - 1]
print('batch', batch_idx, '/', len(batches), batch_symbols)

env = os.environ.copy()
env['ENGINE_UNIVERSE_SYMBOLS'] = ','.join(batch_symbols)
env['ENGINE_TOP_N'] = str(len(batch_symbols))
env['ENGINE_RULE_ENGINE_MODE'] = 'feature_native'
env['ENGINE_OPTIMIZATION_GATE_MODES'] = 'gated,ungated'
env['ENGINE_OPTIMIZATION_TIMEFRAMES'] = '1m'
env['ENGINE_OPTIMIZATION_WINDOWS'] = 'all,30d,90d,360d'

print('effective_top_n', env['ENGINE_TOP_N'])

env.setdefault('ENGINE_VALIDATION_LIGHT_MODE', 'true')
env.setdefault('ENGINE_VALIDATION_STRICTNESS', 'fast')
env.setdefault('ENGINE_VALIDATION_GATE_MODES', 'gated,ungated')


In [ ]:
import json

raw_root = repo_root / 'engine' / 'data' / 'raw'

def has_raw_1m(symbol: str) -> bool:
    tf_dir = raw_root / f'symbol={symbol}' / 'timeframe=1m'
    if not tf_dir.exists():
        return False
    return any(tf_dir.glob('date=*'))

def run_checked(cmd: list[str], name: str) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, env=env, text=True, capture_output=True)
    if proc.returncode != 0:
        print(f'{name}_STDOUT_TAIL:')
        print((proc.stdout or '')[-6000:])
        print(f'{name}_STDERR_TAIL:')
        print((proc.stderr or '')[-6000:])
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout, stderr=proc.stderr)
    return proc

missing_symbols = [s for s in batch_symbols if not has_raw_1m(s)]
if missing_symbols:
    if SKIP_INGEST:
        print('SKIP_INGEST=True but missing raw data; auto-ingesting missing symbols:', missing_symbols)
    else:
        print('Ingesting batch symbols:', missing_symbols)
    for sym in missing_symbols:
        icmd = ['python', '-m', 'engine.src.main', '--mode', 'once', '--optimize', 'off', '--symbol', sym]
        run_checked(icmd, f'INGEST_{sym}')
else:
    print('All batch symbols already have raw 1m data.')

cmd = ['python', '-m', 'engine.src.main', '--mode', 'iterate', '--max-rounds', str(MAX_ROUNDS)]
run_checked(cmd, 'ITERATE')

if RUN_VALIDATE:
    opt_root = repo_root / 'engine' / 'artifacts' / 'optimization' / 'single'
    date_dirs = sorted([d for d in opt_root.iterdir() if d.is_dir() and d.name[:4].isdigit()], key=lambda p: p.name)
    latest_dir = date_dirs[-1] if date_dirs else None
    summary_rel = str((latest_dir / 'summary.json').relative_to(repo_root)) if latest_dir else None
    if summary_rel:
        summary_obj = json.loads((latest_dir / 'summary.json').read_text(encoding='utf-8'))
        if len(summary_obj.get('results', [])) == 0:
            print('Validation skipped: summary has 0 results. Check ingestion/universe coverage.')
        else:
            vcmd = ['python', '-m', 'engine.src.main', '--mode', 'validate', '--summary-path', summary_rel]
            run_checked(vcmd, 'VALIDATE')


In [ ]:
opt_root = repo_root / 'engine' / 'artifacts' / 'optimization' / 'single'
date_dirs = sorted([d for d in opt_root.iterdir() if d.is_dir() and d.name[:4].isdigit()], key=lambda p: p.name)
latest_dir = date_dirs[-1] if date_dirs else None
summary_rel = str((latest_dir / 'summary.json').relative_to(repo_root)) if latest_dir else None
validation_rel = str((latest_dir / 'validation_report.json').relative_to(repo_root)) if latest_dir else None
deploy_rel = str((latest_dir / 'deploy_pool.json').relative_to(repo_root)) if latest_dir else None

manifest_cmd = [
    'python', 'scripts/cloud_dispatch.py', 'manifest',
    '--target', 'kaggle',
    '--status', 'completed',
    '--batch-index', str(batch_idx),
    '--batch-total', str(len(batches)),
    '--symbols', ','.join(batch_symbols),
    '--tasks-total', '1',
    '--tasks-done', '1',
    '--auto-quality',
    '--mirror-to-daily',
]
if summary_rel:
    manifest_cmd.extend(['--summary-path', summary_rel])
if validation_rel:
    manifest_cmd.extend(['--validation-path', validation_rel])
if deploy_rel:
    manifest_cmd.extend(['--deploy-path', deploy_rel])
subprocess.run(manifest_cmd, check=True)
print('manifest written to engine/artifacts/cloud/cloud_run_manifest.json')
